In [1]:
import json
import pickle
import numpy as np
import pandas as pd
import os
print(os.getcwd())

c:\Users\atom0\OneDrive\Documents\College_folder\4th\DACN_Do-Thanh-Thai\CL-TTE\playground


In [3]:
print(os.listdir("../../data/mydata"))

['network_porto', 'porto_data.zip', 'porto_data_2.zip', 'porto_data_3.0.zip', 'test.npy', 'train.npy', 'val.npy']


In [4]:
data_path = "../../data/mydata/"
train_path = os.path.join(data_path, "train.npy")
data = np.load(train_path, allow_pickle=True)
print(f"Loaded data from {train_path}, shape: {data.shape}")

Loaded data from ../../data/mydata/train.npy, shape: (900877, 6)


In [5]:
print(data[0:3])

[[1386940210620000409
  list([10497, 3860, 2554, 2556, 6897, 1945, 1910, 2547, 3889, 9651, 9654, 4074, 9658, 8588, 3941, 9661, 10295, 9671, 3939, 3872, 3935, 6604, 3833, 2520, 3823, 3832, 6592])
  4 347 790 405]
 [1394564373620000261
  list([7821, 353, 4721, 1600, 1591, 3094, 1590, 1092, 3095, 3097, 814, 5259, 818, 52, 54, 58, 22, 9138, 4743, 2929, 4747, 2917, 2919, 7578, 78, 75, 7090, 7297, 10603, 717, 4423, 805, 9596, 789, 7182, 785])
  1 70 1139 825]
 [1402817103620000492
  list([9140, 54, 57, 4748, 6541, 6542, 6539, 6544, 6557, 7222, 770, 768, 2940, 5244, 5245, 767, 858, 156, 10411, 8562, 152, 6867, 7819, 247, 8846, 7421, 4978, 4981, 10550, 10197, 5032, 2268, 1383, 1397, 1399, 1393, 5374, 1396, 670, 3282])
  6 166 445 465]]


In [6]:
y_train = data[:, -1]

In [7]:
import numpy as np

# assuming y_train is a numpy array or tensor of travel times in seconds
y = np.array(y_train)

print(f"mean:   {y.mean():.1f}s  ({y.mean()/60:.1f} min)")
print(f"std:    {y.std():.1f}s")
print(f"median: {np.median(y):.1f}s")
print(f"p10:    {np.percentile(y, 10):.1f}s")
print(f"p25:    {np.percentile(y, 25):.1f}s")
print(f"p75:    {np.percentile(y, 75):.1f}s")
print(f"p90:    {np.percentile(y, 90):.1f}s")

# pairwise distance distribution — sample to avoid O(N^2) cost
idx = np.random.choice(len(y), size=min(2000, len(y)), replace=False)
y_sample = y[idx]
dist = np.abs(y_sample[:, None] - y_sample[None, :])
upper = dist[np.triu_indices(len(y_sample), k=1)]

print(f"\npairwise |yi - yj|:")
print(f"  mean:   {upper.mean():.1f}s")
print(f"  p10:    {np.percentile(upper, 10):.1f}s")
print(f"  p20:    {np.percentile(upper, 20):.1f}s")
print(f"  p30:    {np.percentile(upper, 30):.1f}s")
print(f"  median: {np.median(upper):.1f}s")

mean:   616.8s  (10.3 min)
std:    333.5s
median: 555.0s
p10:    285.0s
p25:    390.0s
p75:    765.0s
p90:    1005.0s

pairwise |yi - yj|:
  mean:   357.0s
  p10:    45.0s
  p20:    90.0s
  p30:    150.0s
  median: 255.0s


In [9]:
r_candidates = [30, 35, 40, 45, 50, 55, 60]
for r in r_candidates:
    pos_rate = (upper <= r).mean()
    print(f"r={r:4d}s → positive rate: {pos_rate:.3f} ({pos_rate*100:.1f}%)")

r=  30s → positive rate: 0.081 (8.1%)
r=  35s → positive rate: 0.081 (8.1%)
r=  40s → positive rate: 0.081 (8.1%)
r=  45s → positive rate: 0.113 (11.3%)
r=  50s → positive rate: 0.113 (11.3%)
r=  55s → positive rate: 0.113 (11.3%)
r=  60s → positive rate: 0.145 (14.5%)


In [5]:
with open(os.path.join(data_path,"nwk_hcm/hcm_edges_poi_new_simplify.pkl"), 'rb') as f:
    edgeinfo = pickle.load(f)
with open(os.path.join(data_path,"nwk_hcm/hcm_nodes_new.pkl"), 'rb') as f:
    nodeinfo = pickle.load(f)

In [6]:
import torch

In [7]:
highway = {'<PAD>': 0, 'unclassified': 1, 'busway': 2, 'crossing': 3, 'living_street': 4, 'motorway': 5, 'motorway_link': 6, 'primary': 7, 'primary_link': 8, 'residential': 9, 'road': 10, 'secondary': 11, 'secondary_link': 12, 'tertiary': 13, 'tertiary_link': 14, 'trunk': 15, 'trunk_link': 16}
def augment_segments(seg, n_poi_groups,
                     p_highway=0.2,
                     p_poi=0.35,
                     p_seg=0.12):

    seg = seg.copy()  # (T, F)

    # --- Highway dropout ---
    highway = seg[:, :2]
    mask_hw = np.random.rand(*highway.shape) < p_highway
    highway[mask_hw] = 1  # unclassified
    seg[:, :2] = highway

    # --- POI dropout ---
    poi = seg[:, 8:]
    mask_poi = np.random.rand(*poi.shape) < p_poi
    poi = poi * (~mask_poi)
    seg[:, 8:] = poi

    # --- Segment dropout (feature masking, NOT removal) ---
    T = seg.shape[0]
    seg_mask = np.random.rand(T) < p_seg

    # prevent full collapse
    if seg_mask.all():
        seg_mask[np.random.randint(T)] = False

    for i in range(T):
        if seg_mask[i]:
            seg[i, 0:2] = 1        # highway → unclassified
            seg[i, 4:8] *= 0.3     # GPS  
            seg[i, 8:]  *= 0.3     # POI 
            # KEEP length (2) and cumlen (3)

    return seg

def parse_highway_tags(raw_val, max_tags=2):
    """Converts OSM strings/lists to a fixed-size list of IDs."""
    UNCLASSIFIED_ID = highway.get('unclassified', 1)
    
    # 1. Handle string/list input
    if isinstance(raw_val, str) and raw_val.startswith("["):
        try: tags = ast.literal_eval(raw_val)
        except: tags = [raw_val]
    elif isinstance(raw_val, list):
        tags = raw_val
    else:
        tags = [raw_val]

    # 2. Map to IDs with fallback
    ids = [highway.get(t, UNCLASSIFIED_ID) for t in tags]
    
    # 3. Pad with 0 (Reserved for 'No Tag')
    while len(ids) < max_tags:
        ids.append(0)
    return ids[:max_tags]


In [8]:

def collate_func(data):

    time = torch.Tensor([d[-1] for d in data])
    linkids = [np.asarray(l[1]) for l in data]
    dateinfo = []
    inds = []
    n_poi_groups = 9
    # 1. Date/Time Preprocessing
    for l in data:
        wday = int(l[2])
        doy_norm = (float(l[3]) / 365.0) * 2 * np.pi
        minute_norm = (float(l[4]) / 1440.0) * 2 * np.pi
        dateinfo.append([wday, doy_norm, minute_norm])
        inds.append(l[0])
    
    lens = np.asarray([len(k) for k in linkids], dtype=np.int16)
    max_seq_len = lens.max()
    def get_infos(xs):
        infos = []
        for x in xs:
            info = edgeinfo[x]
            infot = []
            
            # --- HIGHWAY: Now returns 2 IDs instead of 1 ---
            infot += parse_highway_tags(info[0]) # Adds [ID1, ID2]
            
            infot.append(info[1]) # Length
            infot.append(0)       # Placeholder for cumulative length (calculated later)
            
            try:
                infot += [nodeinfo[info[2]][0], nodeinfo[info[2]][1], 
                          nodeinfo[info[3]][0], nodeinfo[info[3]][1]]
            except:
                infot += [0.0, 0.0, 0.0, 0.0]
            # poi features
            infot += info[4:]
            infos.append(np.asarray(infot))
        return infos

    # 4. Global Scaling Logic
    # We extract all segments to scale length and GPS coordinates uniformly
    all_segments = []
    for b in linkids:
        seg_list = get_infos(b)
        # Calculate cumulative length within the sequence
        cum_len = 0
        for s in seg_list:
            s[3] = cum_len # Index 3 is the placeholder for cumulative length
            cum_len += s[2] # Index 2 is length
        all_segments.extend(seg_list)
    
    all_segments = np.array(all_segments)

    all_segments = np.nan_to_num(all_segments, 0.0)
    all_segments[:, 8:] = np.maximum(all_segments[:, 8:], 0)
    # 5. Final Padded Tensor Construction
    # Shape: [Batch, Max_Seq, 8] 
    # Features: [HighwayID1, HighwayID2, Len, CumLen, Lat1, Lon1, Lat2, Lon2]
    feature_dim = all_segments.shape[1]
    padded_clean = np.zeros((len(data), max_seq_len, feature_dim), dtype=np.float32)
    padded_aug   = np.zeros_like(padded_clean)
    
    curr_idx = 0
    for i, l in enumerate(lens):
        seg = all_segments[curr_idx : curr_idx + l]
        
        if seg.shape[0] != l:
            print(f"Mismatch at batch {i}: expected {l}, got {seg.shape[0]}")
            
        padded_clean[i, :l] = seg
        

        # augmented view
        seg_aug = augment_segments(seg, n_poi_groups)
        padded_aug[i, :l] = seg_aug

        curr_idx += l
    
    return {
        'links_clean': torch.from_numpy(padded_clean),
        'links_aug': torch.from_numpy(padded_aug),
        'dateinfo': torch.from_numpy(np.asarray(dateinfo, dtype=np.float32)),
        'lens': torch.LongTensor(lens), 
        'inds': inds, 
    }, time


In [ ]:
collect_data = []
count = 0
for d in data:
    if count <= 32:
        collect_data.append(d)
        count += 1
        continue
    else:
        d_coll = collate_func(collect_data)
        print(d_coll[0]['links_clean'].shape)
        print(d_coll[0]['links_aug'].shape)
        print(d_coll[0]['dateinfo'].shape)
        print(d_coll[0]['lens'].shape)
        print(d_coll[1].shape)
        
        collect_data = []
        count = 0

torch.Size([33, 178, 18])
torch.Size([33, 178, 18])
torch.Size([33, 3])
torch.Size([33])
torch.Size([33])
torch.Size([33, 200, 18])
torch.Size([33, 200, 18])
torch.Size([33, 3])
torch.Size([33])
torch.Size([33])
torch.Size([33, 187, 18])
torch.Size([33, 187, 18])
torch.Size([33, 3])
torch.Size([33])
torch.Size([33])
torch.Size([33, 196, 18])
torch.Size([33, 196, 18])
torch.Size([33, 3])
torch.Size([33])
torch.Size([33])
torch.Size([33, 153, 18])
torch.Size([33, 153, 18])
torch.Size([33, 3])
torch.Size([33])
torch.Size([33])
torch.Size([33, 164, 18])
torch.Size([33, 164, 18])
torch.Size([33, 3])
torch.Size([33])
torch.Size([33])
torch.Size([33, 215, 18])
torch.Size([33, 215, 18])
torch.Size([33, 3])
torch.Size([33])
torch.Size([33])
torch.Size([33, 145, 18])
torch.Size([33, 145, 18])
torch.Size([33, 3])
torch.Size([33])
torch.Size([33])
torch.Size([33, 209, 18])
torch.Size([33, 209, 18])
torch.Size([33, 3])
torch.Size([33])
torch.Size([33])
torch.Size([33, 201, 18])
torch.Size([33, 201,

In [ ]:
import numpy as np
from tqdm import tqdm

In [ ]:
import numpy as np

all_edge_lengths = []
all_culm_lengths = []

for d in tqdm(data):
    edge_list = d[1]
    cum = 0
    for eid in edge_list:
        edge_len = edgeinfo[eid][1]
        all_edge_lengths.append(edge_len)
        
        cum += edge_len
        all_culm_lengths.append(cum)

# convert to numpy arrays
all_edge_lengths = np.array(all_edge_lengths)
all_culm_lengths = np.array(all_culm_lengths)

# compute mean and std
edge_mean = all_edge_lengths.mean()
edge_std = all_edge_lengths.std()

culm_mean = all_culm_lengths.mean()
culm_std = all_culm_lengths.std()

print(f"Edge length: mean={edge_mean:.3f}, std={edge_std:.3f}")
print(f"Cumulative length: mean={culm_mean:.3f}, std={culm_std:.3f}")

In [ ]:
from tqdm import tqdm
import math

# Initialize
count = 0
start_lat_mean = start_lat_M2 = 0.0
start_lon_mean = start_lon_M2 = 0.0
end_lat_mean = end_lat_M2 = 0.0
end_lon_mean = end_lon_M2 = 0.0

for d in tqdm(data):
    for eid in d[1]:
        edge = edgeinfo[eid]
        s_lat, s_lon, _ = nodeinfo[edge[2]]
        e_lat, e_lon, _ = nodeinfo[edge[3]]
        
        # Increment count
        count += 1
        
        # Start latitude
        delta = s_lat - start_lat_mean
        start_lat_mean += delta / count
        start_lat_M2 += delta * (s_lat - start_lat_mean)
        
        # Start longitude
        delta = s_lon - start_lon_mean
        start_lon_mean += delta / count
        start_lon_M2 += delta * (s_lon - start_lon_mean)
        
        # End latitude
        delta = e_lat - end_lat_mean
        end_lat_mean += delta / count
        end_lat_M2 += delta * (e_lat - end_lat_mean)
        
        # End longitude
        delta = e_lon - end_lon_mean
        end_lon_mean += delta / count
        end_lon_M2 += delta * (e_lon - end_lon_mean)

# Compute standard deviations
start_lat_std = math.sqrt(start_lat_M2 / count)
start_lon_std = math.sqrt(start_lon_M2 / count)
end_lat_std = math.sqrt(end_lat_M2 / count)
end_lon_std = math.sqrt(end_lon_M2 / count)

print(f"Start latitude: mean={start_lat_mean:.6f}, std={start_lat_std:.6f}")
print(f"Start longitude: mean={start_lon_mean:.6f}, std={start_lon_std:.6f}")
print(f"End latitude: mean={end_lat_mean:.6f}, std={end_lat_std:.6f}")
print(f"End longitude: mean={end_lon_mean:.6f}, std={end_lon_std:.6f}")

In [3]:
print(data[0])

[19497
 list([21264, 4999, 21263, 26818, 5309, 43114, 25057, 5690, 43455, 21314, 25072, 25074, 21302, 25133, 21299, 12074, 42247, 42248, 42250, 42252, 5449, 18145, 21309, 44724, 25823, 1194, 21863, 21857, 21368, 21257, 22187, 45642, 22166, 26287, 39457, 26277, 3565, 4710, 14057, 25818, 4433, 21258, 44439, 11330, 11329, 3909, 10039, 10037, 9454, 32938, 10040, 17554, 532, 46138, 40651, 45702, 12573, 45257, 12577, 45247, 12579, 45238, 45233, 12571, 12563, 12568, 40410, 12566, 17982, 45216, 45250, 2074, 9655, 45262, 9649, 10118, 10068, 40374, 26662, 4562, 26207, 13916, 13915, 17508, 40379, 27623, 22521, 22517, 23062, 16843, 23057, 23063, 23068, 23069, 23074, 22512, 22514, 22531, 16969, 22530, 22536, 22535, 28001, 27374, 12220, 4058, 16802, 11411, 11410, 31626, 12222, 11406, 16793, 30095, 32413, 29987, 30102, 44166, 32415, 20886, 23573, 871, 23570, 29255, 14204, 29243, 2543, 23095, 4690, 28536, 18905, 23108, 37830, 37834, 28533, 20911, 28465, 28476, 16674, 37776, 1504, 189, 37780, 16682, 16

In [ ]:
times = []
for d in data:
    times.append(d[-1])
print(np.mean(times), np.std(times))

In [ ]:
print(type(edgeinfo), len(edgeinfo))

In [ ]:
import ast

# 1. Extract all raw types from your edgeinfo values
raw_types = {v[0] for v in edgeinfo.values()}

# 2. Function to turn "['a', 'b']" or "a" into a flat list ['a', 'b']
def listify_string(val):
    if isinstance(val, str) and val.startswith("["):
        try:
            return ast.literal_eval(val)
        except:
            return [val]
    return [val]

# 3. Flatten everything into a single set of unique "atomic" road types
atomic_types = set()
for t in raw_types:
    atoms = listify_string(t)
    for a in atoms:
        atomic_types.add(a)

# 4. Construct the highway dict
# ID 0: Reserved for Padding
# ID 1: Reserved for 'unclassified' (The Catch-all)
highway = {"<PAD>": 0, "unclassified": 1}

# Add all other types starting from ID 2
current_id = 2
for t in sorted(list(atomic_types)):
    if t != "unclassified":
        highway[t] = current_id
        current_id += 1

print(f"Total Unique Atomic Types: {len(highway)}")
print(highway)

In [ ]:
count = {road_type : 0 for road_type in highway.keys()}

In [ ]:
print(edgeinfo[0])

In [ ]:
for d in data:
    edge_id_list = d[1]
    for edge_id in edge_id_list:
        edge = edgeinfo[edge_id]
        road_type = edge[0]
        if road_type.startswith("["):
            try:
                types = ast.literal_eval(road_type)
                for t in types:
                    if t in count:
                        count[t] += 1
            except:
                if road_type in count:
                    count[road_type] += 1
        else:
            if road_type in count:
                count[road_type] += 1
    
print(count)

In [ ]:
print(len(edgeinfo))

In [ ]:
print("First data sample:")
print(data[0])

In [ ]:
def parse_highway_tags(raw_val, max_tags=2):
    """Converts OSM strings/lists to a fixed-size list of IDs."""
    UNCLASSIFIED_ID = highway.get('unclassified', 1)
    
    # 1. Handle string/list input
    if isinstance(raw_val, str) and raw_val.startswith("["):
        try: tags = ast.literal_eval(raw_val)
        except: tags = [raw_val]
    elif isinstance(raw_val, list):
        tags = raw_val
    else:
        tags = [raw_val]

    # 2. Map to IDs with fallback
    ids = [highway.get(t, UNCLASSIFIED_ID) for t in tags]
    
    # 3. Pad with 0 (Reserved for 'No Tag')
    while len(ids) < max_tags:
        ids.append(0)
    return ids[:max_tags]

In [ ]:
id_ = parse_highway_tags(edgeinfo[0][0])
print("Raw highway tag for edge 0:", edgeinfo[0][0])
print(f"Parsed IDs for edge 0: {id_}")

In [ ]:
print("STD: ", np.std(data[:, -1]))

In [ ]:
os.listdir('../')